# Revised Link Prediction Notebook\n\nThis cleaned notebook addresses reviewer comments with a reproducible pipeline.\n- Comment 3: stronger evaluation (repeated splits, variance, CI, significance tests)\n- Comment 4: leakage-safe Node2Vec generated from training graph per split\n- Comment 6: functional enrichment from local GO mapping (offline)\n- Comment 7: interpretable ego-network visualizations\n\nRun top-to-bottom.\n

In [ ]:
# Reviewer-aligned, reproducible pipeline for Flavobacterium link prediction

from pathlib import Path
import os
import random
import warnings
import platform

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn as nn
import torch.nn.functional as F

from Bio import SeqIO
from sklearn.metrics import roc_auc_score, average_precision_score
from torch_geometric.data import Data
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from torch_geometric.utils import to_undirected, negative_sampling

import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Reviewer Comment 3/4/6/7: reproducibility and unified configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CONFIG = {
    "fasta_path": Path("../Flavobacterium sp. MO_0223_q7L1K.faa"),
    "string_path": Path("../string_interactions.tsv"),
    "mapping_path": Path("../string_mapping.tsv"),
    "biomni_path": Path("../complete_vaccine_analysis_all_3257_proteins.csv"),
    "esm_file": Path("esm2_embeddings.h5"),
    "protein_go_path": Path("../protein_to_go.tsv"),
    "base_seed": 42,
    "num_splits": 5,
    "num_val": 0.1,
    "num_test": 0.1,
    "max_epochs": 60,
    "patience": 12,
    "lr": 1e-3,
    "hidden_dim": 128,
    "dropout": 0.5,
    "node2vec_dim": 128,
    "n2v_walk_length": 10,
    "n2v_num_walks": 80,
    "n2v_window": 5,
    "n2v_iterations": 1,
    "n2v_workers": 4,
    "prediction_threshold": 0.90,
    "priority_top_k": 2000,
    "ego_count": 4,
    "ego_neighbors": 30,
    "edge_score_batch": 250000,
}

FEATURE_SETS = [
    "Biomni only",
    "ESM only",
    "Node2Vec only",
    "Biomni + ESM",
    "Biomni + Node2Vec",
    "ESM + Node2Vec",
    "Biomni + ESM + Node2Vec",
]
MODELS = ["GCN", "GAT", "GraphSAGE"]


def set_global_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [ ]:
# %% CELL 1: Setup and data loading
set_global_seed(CONFIG["base_seed"])
print("Device:", DEVICE)
print("Working directory:", Path.cwd())

required = [
    CONFIG["fasta_path"],
    CONFIG["string_path"],
    CONFIG["mapping_path"],
    CONFIG["biomni_path"],
    CONFIG["esm_file"],
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required file(s):\n" + "\n".join(missing))

fasta_records = list(SeqIO.parse(str(CONFIG["fasta_path"]), "fasta"))
fasta_seq_dict = {rec.id: str(rec.seq) for rec in fasta_records}
print(f"FASTA loaded: {len(fasta_seq_dict)} proteins")

string_df = pd.read_csv(CONFIG["string_path"], sep="\t")
string_df = string_df[["#node1", "node2", "combined_score"]].copy()
string_df["combined_score"] = string_df["combined_score"].astype(float)
print(f"STRING edges loaded: {len(string_df):,}")

G = nx.Graph()
for _, row in string_df.iterrows():
    G.add_edge(row["#node1"], row["node2"], weight=row["combined_score"])

mapping_df = pd.read_csv(CONFIG["mapping_path"], sep="\t")
if "preferredName" not in mapping_df.columns or "queryItem" not in mapping_df.columns:
    raise ValueError("string_mapping.tsv must contain columns: preferredName, queryItem")
string_to_fasta_map = dict(zip(mapping_df["preferredName"], mapping_df["queryItem"]))
G_remapped = nx.relabel_nodes(G, string_to_fasta_map, copy=True)

biomni_df = pd.read_csv(CONFIG["biomni_path"]).set_index("protein_id")
feature_cols = [
    "signal_peptide_strength",
    "number_of_tm_domains",
    "cysteine_content",
    "gravy_hydrophobicity",
    "protein_length",
    "molecular_weight_da",
    "instability_index",
    "isoelectric_point",
    "vaccine_score",
]
priority_map = {"Very Low": 0, "Low": 1, "Medium": 2, "High": 3}

for pid in biomni_df.index:
    if pid not in G_remapped:
        G_remapped.add_node(pid)

missing_biomni = 0
missing_seq = 0
for node in G_remapped.nodes:
    if node in biomni_df.index:
        row = biomni_df.loc[node]
        G_remapped.nodes[node]["features"] = row[feature_cols].astype(float).values
        G_remapped.nodes[node]["priority"] = int(priority_map.get(row["priority"], -1))
    else:
        G_remapped.nodes[node]["features"] = np.zeros(len(feature_cols), dtype=float)
        G_remapped.nodes[node]["priority"] = -1
        missing_biomni += 1

    seq = fasta_seq_dict.get(node)
    G_remapped.nodes[node]["sequence"] = seq
    if seq is None:
        missing_seq += 1

for node in G_remapped.nodes:
    G_remapped.nodes[node]["isolated"] = (G_remapped.degree(node) == 0)

print(f"Graph after remapping + completion: {G_remapped.number_of_nodes():,} nodes, {G_remapped.number_of_edges():,} edges")
print(f"Missing Biomni rows: {missing_biomni}")
print(f"Missing sequences: {missing_seq}")

# ESM embeddings
import h5py
ordered_nodes = list(G_remapped.nodes)
node_to_idx = {n: i for i, n in enumerate(ordered_nodes)}
idx_to_node = {i: n for n, i in node_to_idx.items()}

esm_emb = {}
with h5py.File(CONFIG["esm_file"], "r") as h5:
    keys = set(h5.keys())
    if not keys:
        raise ValueError("ESM H5 file is empty")
    first_key = next(iter(keys))
    esm_dim = int(h5[first_key][:].shape[0])
    for pid in ordered_nodes:
        if pid in h5:
            esm_emb[pid] = h5[pid][:].astype(np.float32)
        else:
            esm_emb[pid] = np.zeros(esm_dim, dtype=np.float32)

X_biomni = []
X_esm = []
for pid in ordered_nodes:
    if pid in biomni_df.index:
        biom = biomni_df.loc[pid][feature_cols].values.astype(float)
    else:
        biom = np.zeros(len(feature_cols), dtype=float)
    X_biomni.append(biom)
    X_esm.append(esm_emb[pid])

X_biomni = torch.tensor(np.vstack(X_biomni), dtype=torch.float)
X_esm = torch.tensor(np.vstack(X_esm), dtype=torch.float)
print("X_biomni:", tuple(X_biomni.shape))
print("X_esm:", tuple(X_esm.shape))


In [ ]:
# %% CELL 2: Model and utility definitions
class GCNLinkPredictor(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.dropout = dropout

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, z, edge_label_index):
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        return (src * dst).sum(dim=-1)

    def forward(self, x, edge_index, edge_label_index):
        return self.decode(self.encode(x, edge_index), edge_label_index)


class GATLinkPredictor(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64, heads=4, dropout=0.5):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden_dim // heads, heads=heads, dropout=dropout)
        self.conv2 = GATConv(hidden_dim, out_dim, heads=1, dropout=dropout)
        self.dropout = dropout

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, z, edge_label_index):
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        return (src * dst).sum(dim=-1)

    def forward(self, x, edge_index, edge_label_index):
        return self.decode(self.encode(x, edge_index), edge_label_index)


class GraphSAGELinkPredictor(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, out_dim=64, dropout=0.5):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim)
        self.conv2 = SAGEConv(hidden_dim, out_dim)
        self.dropout = dropout

    def encode(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.conv2(x, edge_index)
        return x

    def decode(self, z, edge_label_index):
        src = z[edge_label_index[0]]
        dst = z[edge_label_index[1]]
        return (src * dst).sum(dim=-1)

    def forward(self, x, edge_index, edge_label_index):
        return self.decode(self.encode(x, edge_index), edge_label_index)


MODEL_MAP = {
    "GCN": GCNLinkPredictor,
    "GAT": GATLinkPredictor,
    "GraphSAGE": GraphSAGELinkPredictor,
}

bce_loss = nn.BCEWithLogitsLoss()


def train_one_epoch(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    logits = model(data.x.to(DEVICE), data.edge_index.to(DEVICE), data.edge_label_index.to(DEVICE))
    y = data.edge_label.to(DEVICE).float()
    loss = bce_loss(logits, y)
    loss.backward()
    optimizer.step()
    return float(loss.item())


@torch.no_grad()
def eval_auc_ap(model, data):
    model.eval()
    logits = model(data.x.to(DEVICE), data.edge_index.to(DEVICE), data.edge_label_index.to(DEVICE))
    y_true = data.edge_label.detach().cpu().numpy()
    y_score = torch.sigmoid(logits).detach().cpu().numpy()
    if len(np.unique(y_true)) < 2:
        return 0.5, 0.5
    auc = roc_auc_score(y_true, y_score)
    ap = average_precision_score(y_true, y_score)
    return float(auc), float(ap)


def run_single_model(model_name, in_dim, train_data, val_data, test_data, split_seed):
    # Reviewer Comment 3: fixed random seed per split
    set_global_seed(split_seed)

    model = MODEL_MAP[model_name](
        in_dim=in_dim,
        hidden_dim=CONFIG["hidden_dim"],
        dropout=CONFIG["dropout"],
    ).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

    best = {
        "best_val_auc": 0.0,
        "test_auc": 0.0,
        "test_ap": 0.0,
        "best_epoch": 0,
        "best_state": None,
    }
    no_improve = 0

    for epoch in range(1, CONFIG["max_epochs"] + 1):
        _ = train_one_epoch(model, train_data, optimizer)
        val_auc, _ = eval_auc_ap(model, val_data)

        if val_auc > best["best_val_auc"] + 1e-4:
            test_auc, test_ap = eval_auc_ap(model, test_data)
            best["best_val_auc"] = val_auc
            best["test_auc"] = test_auc
            best["test_ap"] = test_ap
            best["best_epoch"] = epoch
            best["best_state"] = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= CONFIG["patience"]:
                break

    return best


def _node2vec_with_karateclub(train_graph, dim, seed):
    from karateclub import Node2Vec as KCNode2Vec

    str_graph = nx.relabel_nodes(train_graph, {n: str(n) for n in train_graph.nodes()})
    model = KCNode2Vec(
        dimensions=dim,
        walk_length=CONFIG["n2v_walk_length"],
        num_walks=CONFIG["n2v_num_walks"],
        window_size=CONFIG["n2v_window"],
        workers=CONFIG["n2v_workers"],
        iterations=CONFIG["n2v_iterations"],
        seed=seed,
    )
    model.fit(str_graph)
    return {int(node): emb for node, emb in zip(model.get_members(), model.get_embedding())}


def _node2vec_with_gensim(train_graph, dim, seed):
    # Reviewer Comment 4: fallback for Mac environments where karateclub may be unavailable
    from gensim.models import Word2Vec

    random.seed(seed)
    np.random.seed(seed)

    walk_length = max(10, CONFIG["n2v_walk_length"])
    num_walks = max(10, CONFIG["n2v_num_walks"])
    nodes = list(train_graph.nodes())

    def random_walk(start):
        walk = [start]
        for _ in range(walk_length - 1):
            cur = walk[-1]
            nbrs = list(train_graph.neighbors(cur))
            if not nbrs:
                break
            walk.append(random.choice(nbrs))
        return [str(x) for x in walk]

    walks = []
    for _ in range(num_walks):
        random.shuffle(nodes)
        for n in nodes:
            walks.append(random_walk(n))

    w2v = Word2Vec(
        sentences=walks,
        vector_size=dim,
        window=CONFIG["n2v_window"],
        min_count=1,
        sg=1,
        workers=CONFIG["n2v_workers"],
        epochs=5,
        seed=seed,
    )

    out = {}
    for n in nodes:
        key = str(n)
        out[n] = w2v.wv[key] if key in w2v.wv else np.zeros(dim, dtype=np.float32)
    return out


def generate_node2vec_tensor_from_train_edges(num_nodes, train_edge_index, seed):
    # Reviewer Comment 4: train-graph-only Node2Vec to avoid leakage from held-out edges
    train_graph = nx.Graph()
    train_graph.add_nodes_from(range(num_nodes))
    train_graph.add_edges_from(train_edge_index.t().cpu().numpy().tolist())

    dim = CONFIG["node2vec_dim"]
    try:
        emb_map = _node2vec_with_karateclub(train_graph, dim=dim, seed=seed)
        backend = "karateclub"
    except Exception:
        emb_map = _node2vec_with_gensim(train_graph, dim=dim, seed=seed)
        backend = "gensim-fallback"

    mat = np.zeros((num_nodes, dim), dtype=np.float32)
    for i in range(num_nodes):
        if i in emb_map:
            mat[i] = np.asarray(emb_map[i], dtype=np.float32)

    print(f"Node2Vec backend: {backend}")
    return torch.tensor(mat, dtype=torch.float)


def compose_features(feature_name, n2v_tensor=None):
    if "Node2Vec" in feature_name and n2v_tensor is None:
        raise ValueError("Node2Vec feature requested but n2v_tensor is None")

    if feature_name == "Biomni only":
        return X_biomni
    if feature_name == "ESM only":
        return X_esm
    if feature_name == "Node2Vec only":
        return n2v_tensor
    if feature_name == "Biomni + ESM":
        return torch.cat([X_biomni, X_esm], dim=1)
    if feature_name == "Biomni + Node2Vec":
        return torch.cat([X_biomni, n2v_tensor], dim=1)
    if feature_name == "ESM + Node2Vec":
        return torch.cat([X_esm, n2v_tensor], dim=1)
    if feature_name == "Biomni + ESM + Node2Vec":
        return torch.cat([X_biomni, X_esm, n2v_tensor], dim=1)

    raise ValueError(f"Unknown feature set: {feature_name}")


In [ ]:
# %% CELL 3: Repeated-split experiments, summary, stats, and figures
edge_pairs = [[node_to_idx[u], node_to_idx[v]] for u, v in G_remapped.edges()]
edge_index = torch.tensor(edge_pairs, dtype=torch.long).t().contiguous()
edge_index = to_undirected(edge_index, num_nodes=len(ordered_nodes))

full_data = Data(
    x=torch.zeros((len(ordered_nodes), 1), dtype=torch.float),
    edge_index=edge_index,
)


def run_one_split(split_id, split_seed):
    set_global_seed(split_seed)

    splitter = RandomLinkSplit(
        num_val=CONFIG["num_val"],
        num_test=CONFIG["num_test"],
        is_undirected=True,
        add_negative_train_samples=True,
        split_labels=False,
    )
    train_data, val_data, test_data = splitter(full_data)

    n2v_tensor = None
    if any("Node2Vec" in fs for fs in FEATURE_SETS):
        n2v_tensor = generate_node2vec_tensor_from_train_edges(
            num_nodes=full_data.num_nodes,
            train_edge_index=train_data.edge_index,
            seed=split_seed,
        )

    split_records = []
    for feature_name in FEATURE_SETS:
        X = compose_features(feature_name, n2v_tensor=n2v_tensor)

        tr = train_data.clone(); tr.x = X
        va = val_data.clone(); va.x = X
        te = test_data.clone(); te.x = X

        for model_name in MODELS:
            best = run_single_model(
                model_name=model_name,
                in_dim=X.shape[1],
                train_data=tr,
                val_data=va,
                test_data=te,
                split_seed=split_seed,
            )
            split_records.append({
                "split_id": split_id,
                "seed": split_seed,
                "feature_set": feature_name,
                "model": model_name,
                "best_val_auc": best["best_val_auc"],
                "test_auc": best["test_auc"],
                "test_ap": best["test_ap"],
                "best_epoch": best["best_epoch"],
            })

    return split_records


all_records = []
for i in range(CONFIG["num_splits"]):
    seed = CONFIG["base_seed"] + i
    print(f"\\n=== Split {i+1}/{CONFIG['num_splits']} (seed={seed}) ===")
    all_records.extend(run_one_split(split_id=i + 1, split_seed=seed))

results_long = pd.DataFrame(all_records)
results_long.to_csv("kfold_results_detailed.csv", index=False)
print("Saved: kfold_results_detailed.csv")

summary = (
    results_long
    .groupby(["feature_set", "model"], as_index=False)
    .agg(
        n=("test_auc", "size"),
        auc_mean=("test_auc", "mean"),
        auc_std=("test_auc", "std"),
        ap_mean=("test_ap", "mean"),
        ap_std=("test_ap", "std"),
        val_auc_mean=("best_val_auc", "mean"),
        val_auc_std=("best_val_auc", "std"),
    )
)
summary["auc_ci95"] = 1.96 * summary["auc_std"].fillna(0.0) / np.sqrt(summary["n"])
summary["ap_ci95"] = 1.96 * summary["ap_std"].fillna(0.0) / np.sqrt(summary["n"])
summary = summary.sort_values(["auc_mean", "ap_mean"], ascending=False).reset_index(drop=True)
summary.to_csv("kfold_results_summary.csv", index=False)

compat = summary[["feature_set", "model", "auc_mean", "auc_std", "ap_mean", "ap_std"]].copy()
compat.columns = ["Feature Set", "Model", "AUC Mean", "AUC Std", "AP Mean", "AP Std"]
compat.to_csv("kfold_results.csv", index=False)
print("Saved: kfold_results_summary.csv")
print("Saved: kfold_results.csv")
display(summary.head(15))

# Reviewer Comment 3: paired significance tests
try:
    from scipy import stats
except Exception as exc:
    raise ImportError("scipy is required for significance testing. Install with: pip install scipy") from exc

pivot_auc = results_long.pivot_table(index="split_id", columns=["feature_set", "model"], values="test_auc")
best_row = summary.iloc[0]
best_key = (best_row["feature_set"], best_row["model"])
best_scores = pivot_auc[best_key]

stat_rows = []
for col in pivot_auc.columns:
    if col == best_key:
        continue
    paired = pd.concat([best_scores, pivot_auc[col]], axis=1).dropna()
    if len(paired) < 2:
        continue
    x = paired.iloc[:, 0].values
    y = paired.iloc[:, 1].values
    _, t_p = stats.ttest_rel(x, y)
    try:
        _, w_p = stats.wilcoxon(x, y, zero_method="wilcox", alternative="two-sided")
    except Exception:
        w_p = np.nan

    stat_rows.append({
        "best_feature_set": best_key[0],
        "best_model": best_key[1],
        "comp_feature_set": col[0],
        "comp_model": col[1],
        "mean_diff_auc": float(np.mean(x - y)),
        "paired_t_p": float(t_p),
        "wilcoxon_p": float(w_p) if pd.notna(w_p) else np.nan,
        "n_pairs": int(len(paired)),
    })

stat_df = pd.DataFrame(stat_rows).sort_values("paired_t_p")
stat_df.to_csv("statistical_tests.csv", index=False)
print("Best configuration by mean test AUC:", best_key)
print("Saved: statistical_tests.csv")
display(stat_df.head(20))

# Figures from aggregated split-level results
heat = summary.pivot(index="model", columns="feature_set", values="auc_mean")
plt.figure(figsize=(12, 4.5))
sns.heatmap(heat, annot=True, fmt=".3f", cmap="YlGnBu", linewidths=0.5)
plt.title("Mean Test AUC Across Feature Sets and GNN Models (Repeated Splits)")
plt.xlabel("Feature Set")
plt.ylabel("Model")
plt.tight_layout()
plt.savefig("Test AUC Heatmap Across Models.png", dpi=300)
plt.show()

gsage = summary[summary["model"] == "GraphSAGE"].copy().sort_values("auc_mean", ascending=False)
plt.figure(figsize=(11, 4.5))
plt.bar(
    gsage["feature_set"],
    gsage["auc_mean"],
    yerr=gsage["auc_std"].fillna(0.0),
    color="#3a6ea5",
    alpha=0.9,
    capsize=4,
)
plt.xticks(rotation=30, ha="right")
plt.ylabel("Test AUC (mean +/- std)")
plt.title("GraphSAGE Performance by Feature Set")
plt.tight_layout()
plt.savefig("Figure_3_Bar_Comparison.png", dpi=300)
plt.show()


In [ ]:
# %% CELL 4: Final predictions, improved network view, enrichment

def compose_final_feature_tensor(feature_name):
    if "Node2Vec" in feature_name:
        full_n2v = generate_node2vec_tensor_from_train_edges(
            num_nodes=len(ordered_nodes),
            train_edge_index=edge_index,
            seed=CONFIG["base_seed"],
        )
    else:
        full_n2v = None
    return compose_features(feature_name, n2v_tensor=full_n2v)


best_cfg = summary.iloc[0]
best_feature_set = best_cfg["feature_set"]
best_model_name = best_cfg["model"]
print("Best config for final interpretation:", best_feature_set, "|", best_model_name)

X_final = compose_final_feature_tensor(best_feature_set)
pos_edges = torch.tensor([[node_to_idx[u], node_to_idx[v]] for u, v in G_remapped.edges()], dtype=torch.long).t().contiguous()
msg_edge_index = to_undirected(pos_edges, num_nodes=len(ordered_nodes))

model = MODEL_MAP[best_model_name](in_dim=X_final.shape[1], hidden_dim=CONFIG["hidden_dim"], dropout=CONFIG["dropout"]).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG["lr"])

model.train()
for epoch in range(1, CONFIG["max_epochs"] + 1):
    optimizer.zero_grad()
    neg_edges = negative_sampling(
        edge_index=msg_edge_index,
        num_nodes=len(ordered_nodes),
        num_neg_samples=pos_edges.shape[1],
        method="sparse",
    )
    edge_label_index = torch.cat([pos_edges, neg_edges], dim=1)
    edge_label = torch.cat([torch.ones(pos_edges.shape[1]), torch.zeros(neg_edges.shape[1])]).to(DEVICE)

    logits = model(X_final.to(DEVICE), msg_edge_index.to(DEVICE), edge_label_index.to(DEVICE))
    loss = bce_loss(logits, edge_label.float())
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Final training epoch {epoch}/{CONFIG['max_epochs']} | loss={loss.item():.4f}")

model.eval()
with torch.no_grad():
    z_final = model.encode(X_final.to(DEVICE), msg_edge_index.to(DEVICE))

isolated_nodes = [n for n, d in G_remapped.nodes(data=True) if d.get("isolated")]
connected_nodes = [n for n in G_remapped.nodes if n not in set(isolated_nodes)]

candidate_pairs = []
for iso in isolated_nodes:
    for conn in connected_nodes:
        if not G_remapped.has_edge(iso, conn):
            candidate_pairs.append((node_to_idx[iso], node_to_idx[conn]))
for i in range(len(isolated_nodes)):
    for j in range(i + 1, len(isolated_nodes)):
        u = isolated_nodes[i]
        v = isolated_nodes[j]
        if not G_remapped.has_edge(u, v):
            candidate_pairs.append((node_to_idx[u], node_to_idx[v]))

candidate_edges = torch.tensor(candidate_pairs, dtype=torch.long).t().contiguous()
print("Candidate edges:", candidate_edges.shape[1])


def score_edges_in_batches(z, edge_idx, batch_size):
    all_scores = []
    for start in range(0, edge_idx.shape[1], batch_size):
        end = min(start + batch_size, edge_idx.shape[1])
        batch = edge_idx[:, start:end].to(DEVICE)
        src = z[batch[0]]
        dst = z[batch[1]]
        logits = (src * dst).sum(dim=-1)
        probs = torch.sigmoid(logits).detach().cpu()
        all_scores.append(probs)
    return torch.cat(all_scores, dim=0)


probs = score_edges_in_batches(z_final, candidate_edges, CONFIG["edge_score_batch"])

pred_df = pd.DataFrame({
    "u_idx": candidate_edges[0].cpu().numpy(),
    "v_idx": candidate_edges[1].cpu().numpy(),
    "score": probs.numpy(),
})
pred_df["u"] = pred_df["u_idx"].map(idx_to_node)
pred_df["v"] = pred_df["v_idx"].map(idx_to_node)
pred_df["u_priority"] = pred_df["u"].map(lambda x: G_remapped.nodes[x]["priority"])
pred_df["v_priority"] = pred_df["v"].map(lambda x: G_remapped.nodes[x]["priority"])
pred_df = pred_df.sort_values("score", ascending=False).reset_index(drop=True)
pred_df.to_csv("predicted_edges_best_model.csv", index=False)

strong_df = pred_df[pred_df["score"] >= CONFIG["prediction_threshold"]].copy()
if len(strong_df) > CONFIG["priority_top_k"]:
    strong_df = strong_df.head(CONFIG["priority_top_k"]).copy()

print("Predicted edges saved: predicted_edges_best_model.csv")
print(f"Strong edges used for priority analysis: {len(strong_df):,}")

priority_labels = {0: "Very Low", 1: "Low", 2: "Medium", 3: "High"}
heat = pd.DataFrame(0, index=list(priority_labels.values()), columns=list(priority_labels.values()))
for _, r in strong_df.iterrows():
    a = int(r["u_priority"])
    b = int(r["v_priority"])
    if a not in priority_labels or b not in priority_labels:
        continue
    A, B = priority_labels[a], priority_labels[b]
    heat.loc[A, B] += 1
    if A != B:
        heat.loc[B, A] += 1

plt.figure(figsize=(7.5, 6))
sns.heatmap(heat, annot=True, fmt="d", cmap="YlGnBu", cbar=True)
plt.title(f"Priority-Class Predicted Links ({best_model_name} | {best_feature_set})")
plt.xlabel("Priority Class")
plt.ylabel("Priority Class")
plt.tight_layout()
plt.savefig("Figure_4_Priority_Link_Heatmap.png", dpi=300)
plt.show()

# Reviewer Comment 7: interpretable ego-network visualization
vis_df = strong_df.head(max(1000, CONFIG["priority_top_k"])).copy()
G_pred = nx.Graph()
for _, r in vis_df.iterrows():
    G_pred.add_edge(r["u"], r["v"], score=float(r["score"]))

for n in G_pred.nodes:
    G_pred.nodes[n]["priority"] = int(G_remapped.nodes[n].get("priority", -1))

high_nodes = [n for n in G_pred.nodes if G_pred.nodes[n]["priority"] == 3]
if not high_nodes:
    high_nodes = sorted(G_pred.nodes, key=lambda n: G_pred.degree(n), reverse=True)

ego_roots = sorted(high_nodes, key=lambda n: G_pred.degree(n), reverse=True)[:CONFIG["ego_count"]]
priority_colors = {0: "#b3b3b3", 1: "#4f81bd", 2: "#f4a261", 3: "#d62828"}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, ego in zip(axes, ego_roots):
    nbrs = sorted(G_pred.neighbors(ego), key=lambda x: G_pred.degree(x), reverse=True)[:CONFIG["ego_neighbors"]]
    sub = G_pred.subgraph([ego] + nbrs).copy()

    pos = nx.spring_layout(sub, seed=CONFIG["base_seed"], k=0.45)
    node_sizes = [220 if n == ego else 80 + 4 * sub.degree(n) for n in sub.nodes]
    node_colors = [priority_colors.get(sub.nodes[n]["priority"], "#999999") for n in sub.nodes]

    nx.draw_networkx_edges(sub, pos, ax=ax, alpha=0.35, width=0.8)
    nx.draw_networkx_nodes(sub, pos, ax=ax, node_size=node_sizes, node_color=node_colors, alpha=0.95)
    nx.draw_networkx_labels(sub, pos, labels={ego: ego}, font_size=7, ax=ax)

    ax.set_title(f"Ego: {ego} | nodes={sub.number_of_nodes()} edges={sub.number_of_edges()}")
    ax.axis("off")

for ax in axes[len(ego_roots):]:
    ax.axis("off")

plt.tight_layout()
plt.savefig("Figure_6_Ego_Networks.png", dpi=300)
plt.show()
print("Saved: Figure_6_Ego_Networks.png")

# Reviewer Comment 6: local GO enrichment (offline)
protein_go_path = CONFIG["protein_go_path"]
if not protein_go_path.exists():
    print(f"Skipping enrichment: local mapping file not found at {protein_go_path}")
else:
    try:
        from scipy.stats import hypergeom
    except Exception as exc:
        raise ImportError("scipy is required for enrichment. Install with: pip install scipy") from exc

    go_df = pd.read_csv(protein_go_path, sep=None, engine="python")
    protein_candidates = ["protein_id", "protein", "gene", "node"]
    go_candidates = ["go_term", "go_id", "go", "term"]
    protein_col = next((c for c in protein_candidates if c in go_df.columns), None)
    go_col = next((c for c in go_candidates if c in go_df.columns), None)

    if protein_col is None or go_col is None:
        raise ValueError(
            "protein_to_go file must include a protein column and a GO column. "
            f"Columns found: {list(go_df.columns)}"
        )

    go_df = go_df[[protein_col, go_col]].dropna().copy()
    go_df.columns = ["protein", "go_term"]

    q = strong_df[(strong_df["u_priority"] >= 2) | (strong_df["v_priority"] >= 2)]
    query_proteins = set(q["u"]).union(set(q["v"]))

    universe_proteins = set(go_df["protein"])
    query = sorted(query_proteins.intersection(universe_proteins))

    if len(query) < 5:
        print("Skipping enrichment: too few query proteins overlap GO mapping.")
    else:
        N = len(universe_proteins)
        n = len(query)
        term_to_prots = go_df.groupby("go_term")["protein"].apply(set).to_dict()

        rows = []
        qset = set(query)
        for term, prots in term_to_prots.items():
            K = len(prots)
            k = len(qset.intersection(prots))
            if k == 0:
                continue
            pval = hypergeom.sf(k - 1, N, K, n)
            rows.append({
                "go_term": term,
                "query_hits": k,
                "term_size": K,
                "query_size": n,
                "universe_size": N,
                "p_value": float(pval),
            })

        enr = pd.DataFrame(rows).sort_values("p_value").reset_index(drop=True)
        m = len(enr)
        if m > 0:
            ranks = np.arange(1, m + 1)
            enr["fdr_bh"] = np.minimum.accumulate((enr["p_value"].values * m / ranks)[::-1])[::-1]
            enr.to_csv("go_enrichment_results.csv", index=False)
            print("Saved: go_enrichment_results.csv")
            display(enr.head(20))

            top = enr.head(15).iloc[::-1]
            plt.figure(figsize=(10, 6))
            plt.barh(top["go_term"], -np.log10(np.clip(top["fdr_bh"], 1e-300, 1.0)), color="#2a9d8f")
            plt.xlabel("-log10(FDR)")
            plt.title("Top Enriched GO Terms (from Predicted Priority-Relevant Proteins)")
            plt.tight_layout()
            plt.savefig("Figure_GO_Enrichment.png", dpi=300)
            plt.show()
        else:
            print("No enriched terms found.")

# Run manifest
import sklearn
import torch_geometric
manifest = {
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_geometric": torch_geometric.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
}
pd.DataFrame([manifest]).to_csv("run_manifest.csv", index=False)
print("Saved: run_manifest.csv")
